In [ ]:
#Week 6, Day 5 — June 26, 2026
#Design Interactive Dashboard Canvas & Filter Controls

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns, os, json, warnings, datetime
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/HCL_Project"
DIRS = {"raw":BASE_DIR+"/data/raw","processed":BASE_DIR+"/data/processed",
        "metadata":BASE_DIR+"/data/metadata","visualizations":BASE_DIR+"/visualizations"}
print("Environment ready ")
# Load all dashboard datasets
master      = pd.read_csv(DIRS['processed']+'/master_table_v2.csv')
predictions = pd.read_csv(DIRS['processed']+'/dashboard_ready_data.csv')
risk_zones  = pd.read_csv(DIRS['processed']+'/risk_zone_summary.csv')
hotspots    = pd.read_csv(DIRS['processed']+'/hotspots.csv')
cluster_df  = pd.read_csv(DIRS['processed']+'/cluster_results.csv')

print(f'master: {master.shape} | predictions: {predictions.shape}')
print(f'risk_zones: {risk_zones.shape} | hotspots: {hotspots.shape}')

#Step 1: Confirm Filter Slicer Values from Real Data

print('FILTER CONTROL VALUES — CONFIRMED FROM ACTUAL DATA')
print('='*60)
print()

# Slicer 1: Country
countries = master['Country'].dropna().unique().tolist()
print(f'Slicer 1 — Country:')
print(f'  Values  : {countries}')
print(f'  Note    : All spatial records are within India maritime zone')
print()

# Slicer 2: Waste Source
waste_sources = master['Waste_Source'].dropna().unique().tolist()
print(f'Slicer 2 — Waste Source:')
print(f'  Values  : {waste_sources}')
print(f'  India   : Consumer_Goods (main), Recycling_Rate=11.5%, Coastal_Risk=High')
print()

# Slicer 3: Ocean Zone (Habitat proxy)
zones = sorted(master['Ocean_Zone'].dropna().unique().tolist())
zone_counts = master['Ocean_Zone'].value_counts()
print(f'Slicer 3 — Ocean Zone (Habitat Type):')
for zone in zones:
    n = zone_counts.get(zone,0)
    print(f'  {zone:<30}: {n:,} master table rows')
print()

# Slicer 4: Risk Cluster
risks = sorted(predictions['Risk_Cluster'].dropna().unique().tolist())
risk_counts = predictions['Risk_Cluster'].value_counts()
print(f'Slicer 4 — Risk Cluster (K-Means):')
for risk in risks:
    n = risk_counts.get(risk,0)
    pct = n/len(predictions)*100
    print(f'  {risk:<15}: {n:,} records ({pct:.1f}%)')
print()

# Slicer 5: Year
years = sorted(master['year'].dropna().unique().astype(int).tolist())
print(f'Slicer 5 — Year:')
print(f'  Range   : {years[0]} – {years[-1]}')
print(f'  Values  : {years}')

## Step 2: Compute Exact KPI Values for Dashboard Tiles

print('DASHBOARD KPI TILE VALUES')
print('='*55)
print()

# KPI 1: Total Plastic Density
plastic_rows = master.dropna(subset=['Plastic_Density_kg_km2'])
total_density = plastic_rows['Plastic_Density_kg_km2'].sum()
print(f'KPI 1 — Total Plastic Density:')
print(f'  Value   : {total_density:.2f} kg/km²  (sum across all bbox grid cells)')
print(f'  Records : {len(plastic_rows)} grid cells have plastic data')
print()

# KPI 2: Species Observations
total_species = master['Species_Count'].sum(skipna=True)
print(f'KPI 2 — Species Observations:')
print(f'  Value   : {int(total_species):,}  (sum of species counts in master table)')
print(f'  Source  : 9,472 cleaned species records across 2015–2026')
print()

# KPI 3: Critical Risk Cells
critical_n   = risk_counts.get('Critical',101)
critical_pct = critical_n/len(predictions)*100
print(f'KPI 3 — Critical Risk Cells:')
print(f'  Value   : {critical_n} records ({critical_pct:.1f}%)')
print(f'  SST     : >30.4°C average (K-Means centroid)')
print(f'  Margin  : only 0.20°C below SST tipping point (30.63°C)')
print()

# KPI 4: SST Tipping Point
print(f'KPI 4 — SST Ecological Tipping Point:')
print(f'  Value   : 30.63°C')
print(f'  Margin  : 2.13°C above current Indian Ocean mean (28.5°C)')
print(f'  Timeline: ~10 years at projected warming rate')

## Step 3: Build Full Dark-Theme Dashboard Canvas

fig = plt.figure(figsize=(24, 15))
fig.patch.set_facecolor('#1a1a2e')

# ── Title ──
fig.text(0.5, 0.97,
         'Ocean Plastic & Marine Biodiversity Risk Dashboard',
         ha='center', va='top', fontsize=20, fontweight='bold', color='white')
fig.text(0.5, 0.93,
         'HCLTech Internship 2026  |  Aditya Saxena  |  North Indian Ocean (Lat 5°N–30°N, Lon 65°E–95°E)',
         ha='center', va='top', fontsize=11, color='#a0a0c0')

# ════════════════════════════════════════
# ROW 1: 4 KPI Tiles
# ════════════════════════════════════════
kpi_data = [
    ('Total Plastic Density',  f'{total_density:.0f} kg/km²', 'Arabian Sea: 35.82%',    '#c0392b', 0.03, 0.80),
    ('Species Observations',   '9,472',                        '2015–2026 total',         '#2980b9', 0.27, 0.80),
    ('Critical Risk Cells',    f'{critical_n} ({critical_pct:.1f}%)', 'SST > 30.4°C', '#e67e22', 0.51, 0.80),
    ('SST Tipping Point',      '30.63°C',                      '2.13°C from current mean','#922b21', 0.75, 0.80),
]
for title, val, sub, color, x, y in kpi_data:
    ax = fig.add_axes([x, y, 0.22, 0.11])
    ax.set_facecolor(color)
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.text(0.5, 0.65, val,   ha='center', va='center', fontsize=17,
            fontweight='bold', color='white', transform=ax.transAxes)
    ax.text(0.5, 0.30, title, ha='center', va='center', fontsize=9,
            color='white', transform=ax.transAxes)
    ax.text(0.5, 0.06, sub,   ha='center', va='center', fontsize=8,
            color='lightyellow', transform=ax.transAxes)
    for s in ax.spines.values(): s.set_edgecolor('white')

# ════════════════════════════════════════
# ROW 2: 3 Main Panels
# ════════════════════════════════════════

# ── Panel A: Risk Zone Pie Chart ──
ax_pie = fig.add_axes([0.03, 0.38, 0.27, 0.38])
ax_pie.set_facecolor('#16213e')
sizes  = [147, 252, 101]
colors = ['#27ae60', '#f39c12', '#e74c3c']
labels = ['Stable\n(n=147, 29.4%)', 'At-Risk\n(n=252, 50.4%)', 'Critical\n(n=101, 20.2%)']
wedges, texts, autotexts = ax_pie.pie(
    sizes, labels=labels, colors=colors, autopct='%1.1f%%',
    startangle=90, textprops={'color':'white','fontsize':8},
    wedgeprops={'edgecolor':'#1a1a2e','linewidth':2}
)
for at in autotexts: at.set_fontsize(9); at.set_fontweight('bold')
ax_pie.set_title('K-Means Risk Zone Distribution\n(Silhouette = 0.2922)',
                 color='white', fontsize=11, fontweight='bold', pad=12)

# ── Panel B: Hotspot Scatter Map ──
ax_map = fig.add_axes([0.34, 0.38, 0.30, 0.38])
ax_map.set_facecolor('#16213e')
threshold_val = plastic_rows['Plastic_Density_kg_km2'].quantile(0.80)
hot = plastic_rows[plastic_rows['Plastic_Density_kg_km2'] >= threshold_val]
non = plastic_rows[plastic_rows['Plastic_Density_kg_km2'] <  threshold_val]
ax_map.scatter(non['grid_lon'], non['grid_lat'], c='#3498db', s=18, alpha=0.45, label='Normal density')
ax_map.scatter(hot['grid_lon'], hot['grid_lat'], c='#e74c3c', s=60, alpha=0.95, label='Hotspot (top 20%)', zorder=5)
# Ocean labels
ax_map.text(70, 17, 'Arabian\nSea',    fontsize=7, color='#a0c4ff', style='italic', alpha=0.9)
ax_map.text(83, 14, 'Bay of\nBengal',  fontsize=7, color='#a0c4ff', style='italic', alpha=0.9)
ax_map.text(91, 12, 'Andaman\nSea',    fontsize=7, color='#a0c4ff', style='italic', alpha=0.9)
ax_map.set_xlim(63, 97); ax_map.set_ylim(3, 31)
ax_map.tick_params(colors='white', labelsize=7)
ax_map.set_xlabel('Longitude (°E)', color='white', fontsize=8)
ax_map.set_ylabel('Latitude (°N)',  color='white', fontsize=8)
ax_map.set_title(f'Plastic Hotspot Map  ({len(hot)} cells)\nTop 20% Density — North Indian Ocean',
                 color='white', fontsize=11, fontweight='bold')
ax_map.legend(fontsize=7, facecolor='#1a1a2e', labelcolor='white', loc='upper right')
for s in ax_map.spines.values(): s.set_edgecolor('#a0a0c0')

# ── Panel C: Species by Risk Zone Bar ──
ax_bar = fig.add_axes([0.68, 0.38, 0.29, 0.38])
ax_bar.set_facecolor('#16213e')
risk_labels = ['Stable', 'At-Risk', 'Critical']
species_vals = [140.67, 118.63, 95.68]
sst_vals     = [27.05,  28.65,  30.42]
bar_colors   = ['#27ae60', '#f39c12', '#e74c3c']
bars = ax_bar.bar(risk_labels, species_vals, color=bar_colors,
                  edgecolor='white', linewidth=0.8, width=0.5)
for bar, sp_val, sst_val in zip(bars, species_vals, sst_vals):
    ax_bar.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5,
                f'{sp_val:.0f} sp.', ha='center', fontsize=10,
                color='white', fontweight='bold')
    ax_bar.text(bar.get_x()+bar.get_width()/2, bar.get_height()/2,
                f'SST\n{sst_val:.1f}°C', ha='center', fontsize=8,
                color='lightyellow', va='center')
ax_bar.set_facecolor('#16213e')
ax_bar.tick_params(colors='white', labelsize=9)
ax_bar.set_ylabel('Avg Species Count\n(K-Means Centroid)', color='white', fontsize=9)
ax_bar.set_title('Species Count by Risk Zone\n(K-Means Cluster Centroids)',
                 color='white', fontsize=11, fontweight='bold')
ax_bar.set_ylim(0, 165)
ax_bar.grid(True, alpha=0.2, axis='y')
for s in ax_bar.spines.values(): s.set_edgecolor('#a0a0c0')

# ════════════════════════════════════════
# ROW 3: Filter Controls Panel
# ════════════════════════════════════════
ax_f = fig.add_axes([0.03, 0.06, 0.94, 0.28])
ax_f.set_facecolor('#0d3b5e')
ax_f.set_xlim(0,1); ax_f.set_ylim(0,1); ax_f.axis('off')
ax_f.text(0.01, 0.90, 'FILTER CONTROLS  (Power BI Slicers)',
          fontsize=13, color='white', fontweight='bold', transform=ax_f.transAxes)
ax_f.text(0.01, 0.78, 'Selecting any slicer value updates ALL panels simultaneously — mentors can slice by threat, region, source, or time.',
          fontsize=8, color='#a0a0c0', transform=ax_f.transAxes)

filter_panels = [
    ('  Country',       'India',
     '(All records within India maritime zone — Arabian Sea, Bay of Bengal, Andaman Sea)',
     '#3498db', 0.01, 0.28),
    ('  Waste Source',  'Consumer_Goods  |  Industrial',
     'India: Consumer_Goods dominant | Recycling Rate: 11.5% | Coastal Risk: High',
     '#2ecc71', 0.35, 0.28),
    ('  Risk Cluster',  'Stable  |  At-Risk  |  Critical',
     'K-Means k=3 | Stable=29.4%, At-Risk=50.4%, Critical=20.2%',
     '#e74c3c', 0.70, 0.28),
    ('  Ocean Zone',    'Arabian_Sea  |  Bay_of_Bengal  |  Andaman_Sea  |  Indian_Ocean  |  Gulf_of_Kutch',
     'Arabian Sea: 35.82% plastic | Bay of Bengal: 27.05%',
     '#e67e22', 0.01, 0.06),
    ('  Year Range',    '2015  →  2026',
     'Plastic: 2015 actual + UNEP projected | Species: 2015–2026 observations',
     '#9b59b6', 0.70, 0.06),
]
for label, vals, note, color, x, y in filter_panels:
    ax_f.text(x,   y+0.24, label, fontsize=10, color=color,
              fontweight='bold', transform=ax_f.transAxes)
    ax_f.text(x,   y+0.12, vals,  fontsize=9,  color='white',
              transform=ax_f.transAxes)
    ax_f.text(x,   y,      note,  fontsize=7,  color='#a0a0c0',
              style='italic', transform=ax_f.transAxes)

plt.savefig(DIRS['visualizations']+'/Week6_Day5_dashboard_canvas.png',
            dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('Dashboard canvas saved ')

## Step 4: Save Filter Control Documentation

filter_doc = DIRS['metadata']+'/dashboard_filter_controls.txt'

with open(filter_doc,'w') as f:
    f.write('DASHBOARD FILTER CONTROLS DOCUMENTATION\n')
    f.write(f'Generated: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}\n')
    f.write('='*60+'\n\n')
    f.write('All 5 slicers are cross-filtering — selecting any value updates\n')
    f.write('all panels (pie, map, bar, KPI tiles) simultaneously.\n\n')

    filter_data = [
        ('Country',       'master_table_v2',      ['India'],
         'All records in India maritime zone'),
        ('Waste_Source',  'master_table_v2',      ['Consumer_Goods'],
         'India main source; Recycling_Rate=11.5%, Coastal_Risk=High'),
        ('Ocean_Zone',    'master_table_v2',
         ['Arabian_Sea','Bay_of_Bengal','Andaman_Sea','Indian_Ocean','Gulf_of_Kutch'],
         'Arabian Sea 35.82% plastic, Bay of Bengal 27.05%'),
        ('Risk_Cluster',  'dashboard_ready_data', ['Stable','At_Risk','Critical'],
         'K-Means k=3 — Stable:29.4%, At-Risk:50.4%, Critical:20.2%'),
        ('year',          'master_table_v2',       [2015],
         'Plastic actual 2015; Species 2015-2026'),
    ]
    for i,(slicer,source,vals,note) in enumerate(filter_data,1):
        f.write(f'Slicer {i}: {slicer}\n')
        f.write(f'  Source table : {source}\n')
        f.write(f'  Values       : {vals}\n')
        f.write(f'  Note         : {note}\n\n')

    f.write('DASHBOARD VISUAL PANELS:\n')
    panels = [
        ('KPI Tile 1', 'Total Plastic Density',   f'{total_density:.0f} kg/km²'),
        ('KPI Tile 2', 'Species Observations',     '9,472 records'),
        ('KPI Tile 3', 'Critical Risk Cells',      f'{critical_n} ({critical_pct:.1f}%)'),
        ('KPI Tile 4', 'SST Tipping Point',        '30.63°C (margin 2.13°C)'),
        ('Panel A',    'Risk Zone Pie',             'K-Means: Stable/At-Risk/Critical'),
        ('Panel B',    'Plastic Hotspot Map',       f'{len(hotspots)} hotspot cells'),
        ('Panel C',    'Species by Risk Zone Bar',  'K-Means centroid species counts'),
        ('Panel D',    'Folium Spatial Map',        'Interactive HTML map (June 27)'),
    ]
    for panel,title,value in panels:
        f.write(f'  {panel:<12} {title:<35} {value}\n')

print(f'Filter control documentation saved ')
print()
# Show file size of canvas
canvas_path = DIRS['visualizations']+'/Week6_Day5_dashboard_canvas.png'
if os.path.exists(canvas_path):
    size_kb = os.path.getsize(canvas_path)/1024
    print(f'Dashboard canvas: {size_kb:.1f} KB saved to /visualizations/')
print()
